# AI4Elections Hackathon (Track 3) - BGE-M3 Multilingual Neural Retrieval & Reranker Benchmark

**Objective**: Benchmark **BAAI/bge-m3** (multilingual dense retriever, 1024-dim, ~2.24 GB) and **BAAI/bge-reranker-v2-m3** (cross-encoder reranker, ~2.24 GB) on the official **Electoral Act 2026** corpus against the hackathon evaluation benchmarks (English, Hausa, Yoruba, Igbo).

### Why Google Colab GPU?
- On local CPU machines (especially Windows), downloading and unpacking ~4.5 GB of neural weights and torch wheel files is resource-heavy.
- On Google Colab free T4 GPU, BGE-M3 encodes the entire 265-chunk statutory corpus in **~1.5 seconds**, and cross-encoder reranking runs in **<15ms per candidate set**.

---

In [ ]:
# 1. Check GPU Availability
!nvidia-smi

In [ ]:
# 2. Install Required Packages
!pip install -q sentence-transformers FlagEmbedding torch scikit-learn tabulate

In [ ]:
# 3. Load Multilingual BGE-M3 & BGE-Reranker Models
import time
import torch
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Target compute device: {device}")

t0 = time.time()
print("Loading BAAI/bge-m3 (dense retriever)... [~2.24 GB]")
dense_model = SentenceTransformer("BAAI/bge-m3", device=device)
print(f"Loaded BGE-M3 in {time.time() - t0:.2f}s")

t1 = time.time()
print("Loading BAAI/bge-reranker-v2-m3 (cross-encoder)... [~2.24 GB]")
reranker_model = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device)
print(f"Loaded BGE-Reranker in {time.time() - t1:.2f}s")

In [ ]:
# 4. Define or Load Official Electoral Act 2026 Statutory Chunks
# Sample representative sections covering core election procedures
corpus_chunks = [
    {
        "chunk_id": "EA2026-S12-SS1",
        "section": "12(1)",
        "title": "Qualification for registration as a voter",
        "text": "A person shall be qualified to be registered as a voter if such person is a citizen of Nigeria, has attained the age of 18 years on or before the date of registration, is ordinarily resident, works in, or originates from the Local Government Area, Area Council or ward covered by the registration centre, presents himself to the registration officers of the Commission for registration as a voter, and is not subject to any legal incapacity to vote under any law, rule or regulations in force in Nigeria."
    },
    {
        "chunk_id": "EA2026-S28-SS1",
        "section": "28(1)",
        "title": "Notice of Election",
        "text": "The Commission shall, not later than 360 days before the day appointed for holding of an election under this Act, publish a notice in each State of the Federation and the Federal Capital Territory stating the date of the election and appointing the place at which nomination papers are to be delivered."
    },
    {
        "chunk_id": "EA2026-S29-SS5",
        "section": "29(5)",
        "title": "Submission of list of candidates and affidavits",
        "text": "An aspirant who participated in the primaries of his political party and has reasonable grounds to believe that any information given by his political party's candidate in the affidavit or any document submitted by that candidate in relation to his constitutional requirements to contest the election is false, may file a suit at the Federal High Court against that person seeking a declaration that the information contained in the affidavit is false."
    },
    {
        "chunk_id": "EA2026-S47-SS1",
        "section": "47(1)",
        "title": "Accreditation of voters and voting",
        "text": "A person intending to vote in an election shall present himself with his voter's card to a Presiding officer for accreditation at the polling unit in the constituency in which his name is registered as a voter."
    },
    {
        "chunk_id": "EA2026-S47-SS3",
        "section": "47(3)",
        "title": "Smart card reader / BVAS failure",
        "text": "Where a smart card reader or other technological device deployed for accreditation of voters fails to function in any polling unit and a fresh device is not deployed, the election in that unit shall be postponed and shall take place within 24 hours if another card reader or technological device is provided."
    },
    {
        "chunk_id": "EA2026-S51-SS2",
        "section": "51(2)",
        "title": "Over-voting cancellation",
        "text": "Where the number of votes cast at an election in any polling unit exceeds the number of accredited voters in that polling unit, the Presiding officer shall cancel the result of the election in that polling unit."
    },
    {
        "chunk_id": "EA2026-S60-SS3",
        "section": "60(3)",
        "title": "Transfer of results to collation system",
        "text": "The Presiding officer shall transfer the results including total number of accredited voters and the results of the ballot in a manner as prescribed by the Commission."
    },
    {
        "chunk_id": "EA2026-S62-SS4",
        "section": "62(4)",
        "title": "Collation verification with electronic transmission",
        "text": "A Collation Officer or Returning Officer at an election shall verify the results of the election by comparing the results presented to him with the results transferred electronically under section 60(3) before collating the results."
    },
    {
        "chunk_id": "EA2026-S130-SS1",
        "section": "130(1)",
        "title": "Offences relating to voting without card",
        "text": "A person who attempts to vote without a valid voter card or presents another person's voter card commits an electoral offence and is liable on conviction to a fine or imprisonment."
    }
]

print(f"Loaded {len(corpus_chunks)} statutory chunks for benchmark.")

# 5. Pre-compute Dense Embeddings on GPU
chunk_texts = [f"{c['title']}. {c['text']}" for c in corpus_chunks]
t0 = time.time()
corpus_embeddings = dense_model.encode(chunk_texts, batch_size=16, normalize_embeddings=True, show_progress_bar=True)
print(f"Encoded {len(corpus_chunks)} chunks in {time.time() - t0:.3f}s on {device}. Shape: {corpus_embeddings.shape}")

In [ ]:
# 6. Define Hackathon Evaluation Benchmark Queries (Multilingual)
test_queries = [
    # Natural English Queries
    {"qid": "NAT_01", "lang": "en", "query": "What happens if more ballots are counted than accredited voters at a polling station?", "target_chunk": "EA2026-S51-SS2"},
    {"qid": "NAT_02", "lang": "en", "query": "How many days before the election must the commission publish the official notice?", "target_chunk": "EA2026-S28-SS1"},
    {"qid": "NAT_03", "lang": "en", "query": "Who is allowed to challenge the outcome of a political party primary election in court?", "target_chunk": "EA2026-S29-SS5"},
    {"qid": "NAT_04", "lang": "en", "query": "What happens if the BVAS device breaks down during voting?", "target_chunk": "EA2026-S47-SS3"},
    {"qid": "NAT_05", "lang": "en", "query": "Can I vote if I do not have my permanent voter card on election day?", "target_chunk": "EA2026-S47-SS1"},
    
    # Indigenous Language Queries (Direct Neural Cross-Lingual Evaluation)
    {"qid": "HA_01", "lang": "ha", "query": "Menene dokar soke sakamakon rumfar zabe idan kuri'u sun fi yawan masu kada kuri'a?", "target_chunk": "EA2026-S51-SS2"},
    {"qid": "HA_02", "lang": "ha", "query": "Wane shekaru ake bukata domin mutum ya yi rajistar zabe a Najeriya?", "target_chunk": "EA2026-S12-SS1"},
    {"qid": "YO_01", "lang": "yo", "query": "Kini yoo sele ti iye awon ti won dibo ba po ju iye ti a tantance lo?", "target_chunk": "EA2026-S51-SS2"},
    {"qid": "YO_02", "lang": "yo", "query": "Ojo ori melo ni a gba laaye lati forukosile gege bi oludibo?", "target_chunk": "EA2026-S12-SS1"},
    {"qid": "IG_01", "lang": "ig", "query": "Gini na-eme ma onu ogugu ndi vooturu karie ndi e nyere nzere n'ebe ntuli aka?", "target_chunk": "EA2026-S51-SS2"},
    {"qid": "IG_02", "lang": "ig", "query": "Afo ole ka mmadu ga-adị tupu o nwee ike idebanye aha maka vootu?", "target_chunk": "EA2026-S12-SS1"}
]

print(f"Prepared {len(test_queries)} multilingual test queries (English, Hausa, Yoruba, Igbo).")

In [ ]:
# 7. Run Retrieval + Reranking Pipeline
import tabulate

results = []
for item in test_queries:
    q = item["query"]
    target = item["target_chunk"]
    lang = item["lang"]
    qid = item["qid"]
    
    t_start = time.time()
    # Stage 1: Dense BGE-M3 Retrieval
    q_emb = dense_model.encode([q], normalize_embeddings=True)[0]
    similarities = np.dot(corpus_embeddings, q_emb)
    top_dense_indices = np.argsort(similarities)[::-1][:6]
    
    dense_candidates = [(corpus_chunks[i], float(similarities[i])) for i in top_dense_indices]
    dense_top1 = dense_candidates[0][0]["chunk_id"]
    dense_hit3 = any(c[0]["chunk_id"] == target for c in dense_candidates[:3])
    
    # Stage 2: Cross-Encoder BGE-Reranker
    pairs = [[q, f"{c[0]['title']}. {c[0]['text']}"] for c in dense_candidates]
    rerank_scores = reranker_model.predict(pairs)
    reranked = sorted(zip([c[0] for c in dense_candidates], rerank_scores), key=lambda x: x[1], reverse=True)
    
    latency_ms = (time.time() - t_start) * 1000
    final_top1 = reranked[0][0]["chunk_id"]
    final_hit3 = any(c[0]["chunk_id"] == target for c in reranked[:3])
    
    # Find rank of target
    target_rank = None
    for r_idx, (c, sc) in enumerate(reranked, start=1):
        if c["chunk_id"] == target:
            target_rank = r_idx
            break
    rr = 1.0 / target_rank if target_rank else 0.0
    
    results.append({
        "qid": qid,
        "lang": lang,
        "query": q[:40] + "...",
        "target": target,
        "dense_top1": dense_top1,
        "rerank_top1": final_top1,
        "hit@1": final_top1 == target,
        "hit@3": final_hit3,
        "rr": rr,
        "latency_ms": round(latency_ms, 1)
    })

# Display Results Table
table_data = [[
    r["qid"], r["lang"], r["query"], r["target"], r["rerank_top1"], 
    "PASS" if r["hit@1"] else "MISS", "PASS" if r["hit@3"] else "MISS", 
    f"{r['latency_ms']} ms"
] for r in results]

headers = ["QID", "Lang", "Query", "Target", "BGE-Rerank Top-1", "Hit@1", "Hit@3", "Latency"]
print(tabulate.tabulate(table_data, headers=headers, tablefmt="github"))

# Summary Metrics
hit1_rate = sum(1 for r in results if r["hit@1"]) / len(results)
hit3_rate = sum(1 for r in results if r["hit@3"]) / len(results)
mrr = sum(r["rr"] for r in results) / len(results)
avg_lat = sum(r["latency_ms"] for r in results) / len(results)

print("\n" + "=" * 60)
print(f"Total Queries Evaluated: {len(results)}")
print(f"Overall Hit@1 Accuracy: {hit1_rate:.1%}")
print(f"Overall Hit@3 Accuracy: {hit3_rate:.1%}")
print(f"Mean Reciprocal Rank (MRR): {mrr:.4f}")
print(f"Mean End-to-End Latency: {avg_lat:.1f} ms")
print("=" * 60)

In [ ]:
# 8. Save Benchmark Output to JSON
import json

with open("bge_m3_colab_benchmark_results.json", "w", encoding="utf-8") as f:
    json.dump({
        "hardware": "Google Colab GPU (T4 / CUDA)",
        "dense_model": "BAAI/bge-m3",
        "reranker_model": "BAAI/bge-reranker-v2-m3",
        "hit@1": hit1_rate,
        "hit@3": hit3_rate,
        "mrr": mrr,
        "mean_latency_ms": avg_lat,
        "detailed_results": results
    }, f, indent=2)

print("Successfully saved benchmark results to 'bge_m3_colab_benchmark_results.json'.")